# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hafsaShaban/flyrank_internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
if not os.path.exists('/content/flyrank_internship'):
    !git clone https://github.com/hafsaShaban/flyrank_internship.git
%cd /content/flyrank_internship

import pandas as pd
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
print("Loaded:", len(df), "rows")

Cloning into 'flyrank_internship'...
remote: Enumerating objects: 207, done.
remote: Counting objects: 100% (207/207), done.
remote: Compressing objects: 100% (163/163), done.
remote: Total 207 (delta 93), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (207/207), 2.41 MiB | 7.70 MiB/s, done.
Resolving deltas: 100% (93/93), done.
/content/flyrank_internship
Loaded: 30000 rows


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [2]:
print("Rows:", len(df))
print("Unique content_id:", df['content_id'].nunique())
print("Unique client_id:", df['client_id'].nunique())

Rows: 30000
Unique content_id: 30000
Unique client_id: 32


**Contract:**

1. **One row =** one piece of content (`content_id`) belonging to one client (`client_id`) — a snapshot of its current SEO/traffic performance and trend status.
2. **Table(s):** `data/raw/content_refresh_anonymized.csv` (single flat table, no joins).
3. **Time window:** no explicit calendar date column exists — the row is a snapshot as of the data pull, with trailing-window signals baked into column names (`impressions_90d`, `clicks_prev_30d`, etc.), all ending at that same snapshot moment.
4. **Predict/rank:** `is_declining_label` — binary flag for whether a page's `trend_direction` is "down". Goal: rank/flag content whose organic performance is declining so it can be prioritized for refresh.
5. **Deliberately excluded:** `provider_used`, `model_used`, `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d` — the `_used` columns describe how content was generated (not a real-world decision-time signal for THIS task), and the `_last_30d` columns overlap suspiciously close to the label's outcome window, risking leakage.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [3]:
feature_cols = ['word_count','char_count','ctr','avg_position','engagement_rate','scroll_rate',
    'ai_traffic_pct','content_age_days','days_since_last_update','freshness_tier','word_count_tier',
    'char_count_tier','impression_tier','position_tier','search_volume','competition','competition_level',
    'cpc','content_type','main_intent','impressions_90d','clicks_90d','pageviews_90d','sessions_90d',
    'users_90d','engaged_sessions_90d','ai_sessions_90d','scroll_events_90d','days_with_impressions',
    'days_with_sessions','impressions_prev_30d','clicks_prev_30d','sessions_prev_30d']
label_cols = ['trend_direction','trend_pct','is_declining_label']
context_cols = ['content_id','client_id']
excluded_cols = ['impressions_last_30d','clicks_last_30d','sessions_last_30d','provider_used','model_used']

all_cols = set(df.columns) | {'is_declining_label'}
classified = set(feature_cols) | set(label_cols) | set(context_cols) | set(excluded_cols)
print("Unclassified columns:", all_cols - classified)
print("Overlap between buckets (should be empty):",
      set(feature_cols) & set(label_cols) & set(context_cols) & set(excluded_cols))

Unclassified columns: {'age_tier_order', 'age_tier'}
Overlap between buckets (should be empty): set()


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
# Grain check
dupes = df.groupby('content_id').size()
print("content_id rows appearing more than once:", (dupes > 1).sum())

# Counts per client
print(df.groupby('client_id').size().describe())

# Missingness overall
print(df.isnull().mean().sort_values(ascending=False).head(10))

# Missingness by content_type (checking for patterned, not random, gaps)
print(df.groupby('content_type')['word_count'].apply(lambda x: x.isnull().mean()))

content_id rows appearing more than once: 0
count      32.000000
mean      937.500000
std      1376.387113
min         3.000000
25%       110.250000
50%       567.000000
75%      1058.750000
max      7008.000000
dtype: float64
provider_used        0.714600
word_count           0.256633
char_count           0.256633
char_count_tier      0.256633
word_count_tier      0.256633
model_used           0.191100
trend_pct            0.112933
competition_level    0.087000
cpc                  0.082267
search_volume        0.082267
dtype: float64
content_type
comparison article    0.000000
feedly article        0.000000
keyword article       0.282979
Name: word_count, dtype: float64


In [5]:
# Query 3: Availability — filter with IS TRUE equivalent (boolean mask)
df['has_position_data'] = df['avg_position'] > 0

total_rows = len(df)
available_rows = df[df['has_position_data'] == True].shape[0]

print("Total rows:", total_rows)
print("Rows with real position data (has_position_data IS TRUE):", available_rows)
print("Survival rate:", round(available_rows / total_rows * 100, 2), "%")

# Row count + span proxy for this "panel" (no explicit date col, so use content_age_days)
print("\nRow count for this slice:", total_rows)
print("content_age_days range:", df['content_age_days'].min(), "-", df['content_age_days'].max())

Total rows: 30000
Rows with real position data (has_position_data IS TRUE): 28795
Survival rate: 95.98 %

Row count for this slice: 30000
content_age_days range: 90 - 564


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [6]:
print("avg_position == 0 (no data) rows:", (df['avg_position'] == 0).sum())
print("Max scroll_rate:", df['scroll_rate'].max())
print("Max ai_traffic_pct:", df['ai_traffic_pct'].max())
print("Sample ctr values:", df['ctr'].head())

avg_position == 0 (no data) rows: 1205
Max scroll_rate: 300.0
Max ai_traffic_pct: 300.0
Sample ctr values: 0    0.76
1    0.05
2    0.09
3    0.49
4    0.13
Name: ctr, dtype: float64


**Named limitation:** this snapshot has no explicit calendar/month column, so I can't independently verify a true date span or partition by month — only trailing-window aggregates (`_90d`, `_prev_30d`) baked into columns. Also, `word_count`/`char_count` are missing in 25.6% of rows, concentrated entirely in `content_type == "keyword article"` — that's a patterned gap, not random, and any feature using word count will be systematically blind to that content type.

## 5. Five features + the leakage trap

In [7]:
feature_frame = df[['content_id', 'client_id',
                     'word_count', 'ctr', 'avg_position',
                     'content_age_days', 'search_volume']].copy()

print(feature_frame.head())

             content_id          client_id  word_count   ctr  avg_position  \
0  content_304f48230142  client_f369cb89fc      3221.0  0.76          10.6   
1  content_a1fb4e703a9e  client_4e07408562      2481.0  0.05          20.3   
2  content_9aa793d4d895  client_7f2253d7e2      3515.0  0.09          36.5   
3  content_331d6c4de07b  client_19581e27de         NaN  0.49           6.2   
4  content_d99b7a2d90ca  client_3fdba35f04      2803.0  0.13          44.0   

   content_age_days  search_volume  
0               187           10.0  
1               445           90.0  
2               141            0.0  
3               463           10.0  
4               263            0.0  


- **word_count** — knowable at the decision moment because it's a static property of the published page, not dependent on future traffic.
- **ctr** — knowable because it's measured from past search impressions/clicks, available before any refresh decision.
- **avg_position** — knowable because it's the page's current/historical SERP rank, observed pre-decision.
- **content_age_days** — knowable because it's just a calendar calculation from publish date to snapshot date.
- **search_volume** — knowable because it's a keyword-level market signal independent of this page's own future performance.

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X = feature_frame.drop(columns=['content_id', 'client_id']).fillna(0)
y = df['is_declining_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Honest baseline
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
honest_score = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print("Honest AUC (no leak):", round(honest_score, 3))

# --- THE TRAP: sneak in a label-derived column ---
X_leak = X.copy()
X_leak['trend_pct'] = df['trend_pct'].fillna(0)  # derived straight from the label!  # derived straight from the label!

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leak, y, test_size=0.2, random_state=42)
model_leak = LogisticRegression(max_iter=1000)
model_leak.fit(X_train_l, y_train_l)
leak_score = roc_auc_score(y_test_l, model_leak.predict_proba(X_test_l)[:, 1])
print("Leaked AUC (with trend_pct):", round(leak_score, 3))

# Delete it, keep the honest number
X_leak = X_leak.drop(columns=['trend_pct'])
print("\nLesson: leaked AUC jumped toward 1.0 because trend_pct IS the label in disguise.")
print("Keeping the honest score:", round(honest_score, 3))

Honest AUC (no leak): 0.588
Leaked AUC (with trend_pct): 1.0

Lesson: leaked AUC jumped toward 1.0 because trend_pct IS the label in disguise.
Keeping the honest score: 0.588


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.